In [ ]:
from pathlib import Path
import json
import pandas as pd

run_dir = Path('/content/drive/MyDrive/projects/santander/data/artifacts/runs/lightgbm-acquisition-v1-20260920T192409Z-46a125e2')
submission = run_dir / 'submission.csv'
manifest = run_dir / 'submission_manifest.json'
generated_template = run_dir / 'generated_sample_submission.csv'
head = pd.read_csv(submission, nrows=3)
print({
    'submission_exists': submission.is_file(),
    'submission_bytes': submission.stat().st_size,
    'columns': head.columns.tolist(),
    'head_rows': head.to_dict(orient='records'),
    'submission_rows': sum(1 for _ in submission.open(encoding='utf-8')) - 1,
    'manifest_exists': manifest.is_file(),
    'template_exists': generated_template.is_file(),
})
print(json.loads(manifest.read_text(encoding='utf-8')))

{'submission_exists': True, 'submission_bytes': 119463810, 'columns': ['ncodpers', 'added_products'], 'head_rows': [{'ncodpers': 15889, 'added_products': 'ind_recibo_ult1 ind_nom_pens_ult1 ind_nomina_ult1 ind_dela_fin_ult1 ind_cno_fin_ult1 ind_ecue_fin_ult1 ind_reca_fin_ult1'}, {'ncodpers': 1170544, 'added_products': 'ind_recibo_ult1 ind_nom_pens_ult1 ind_nomina_ult1 ind_tjcr_fin_ult1 ind_cno_fin_ult1 ind_ecue_fin_ult1 ind_reca_fin_ult1'}, {'ncodpers': 1170545, 'added_products': 'ind_recibo_ult1 ind_nom_pens_ult1 ind_nomina_ult1 ind_cno_fin_ult1 ind_tjcr_fin_ult1 ind_ecue_fin_ult1 ind_reca_fin_ult1'}], 'submission_rows': 929615, 'manifest_exists': True, 'template_exists': True}
{'model_index': '/content/drive/MyDrive/projects/santander/data/artifacts/runs/lightgbm-acquisition-v1-20260920T192409Z-46a125e2/model_artifacts.json', 'run_config': '/content/drive/MyDrive/projects/santander/data/artifacts/runs/lightgbm-acquisition-v1-20260920T192409Z-46a125e2/submission_config_resolved.yaml', 

In [ ]:
from pathlib import Path
import os
import sys
import traceback

repo = Path('/content/Santander-Product-Recommendation')
data_root = Path('/content/drive/MyDrive/projects/santander/data')
os.environ['SANTANDER_DATA_ROOT'] = str(data_root)
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))
for name in list(sys.modules):
    if name == 'src' or name.startswith('src.'):
        del sys.modules[name]

from src.pipeline.lightgbm_v1 import run_lightgbm_v1_competition_from_config
run_dir = data_root / 'artifacts/runs/lightgbm-acquisition-v1-20260920T192409Z-46a125e2'
error_path = run_dir / 'submission_error.txt'
error_path.unlink(missing_ok=True)
try:
    submission = run_lightgbm_v1_competition_from_config(
        run_dir / 'model_artifacts.json', run_dir / 'submission_config_resolved.yaml'
    )
    print({'submission': str(submission), 'exists': submission.is_file(), 'bytes': submission.stat().st_size if submission.is_file() else None})
except Exception:
    error_path.write_text(traceback.format_exc(), encoding='utf-8')
    raise

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[Submission 1/24] scoring ind_ahor_fin_ult1
[Submission 2/24] scoring ind_aval_fin_ult1
[Submission 3/24] scoring ind_cco_fin_ult1
[Submission 4/24] scoring ind_cder_fin_ult1
[Submission 5/24] scoring ind_cno_fin_ult1
[Submission 6/24] scoring ind_ctju_fin_ult1
[Submission 7/24] scoring ind_ctma_fin_ult1
[Submission 8/24] scoring ind_ctop_fin_ult1
[Submission 9/24] scoring ind_ctpp_fin_ult1
[Submission 10/24] scoring ind_deco_fin_ult1
[Submission 11/24] scoring ind_dela_fin_ult1
[Submission 12/24] scoring ind_deme_fin_ult1
[Submission 13/24] scoring ind_ecue_fin_ult1
[Submission 14/24] scoring ind_fond_fin_ult1
[Submission 15/24] scoring ind_hip_fin_ult1
[Submission 16/24] scoring ind_nom_pens_ult1
[Submission 17/24] scoring ind_nomina_ult1
[Submission 18/24] scoring ind_plan_fin_ult1
[Submission 19/24] scoring ind_pres_fin_ult1
[Submission 20/24] scoring ind_reca_fin_ult1
[Submission 21/24] scoring ind_recibo_ult1
[Submission 22/24] scoring ind_tjcr_fin_ult1
[Submission 23/24] scoring

In [ ]:
# Allow a deterministic template generated from test order only when the
# official sample_submission.csv was not supplied with the raw data.
from pathlib import Path
import sys
import yaml

repo = Path('/content/Santander-Product-Recommendation')
pipeline_file = repo / 'src/pipeline/lightgbm_v1.py'
source = pipeline_file.read_text(encoding='utf-8')
if 'import pandas as pd' not in source:
    source = source.replace('import yaml\n', 'import yaml\nimport pandas as pd\n')
old = '''    submission = run_competition_test_inference(
        prepared_frame, root / competition["sample_submission"], artifact_directories,
'''
new = '''    sample_submission = root / competition["sample_submission"]
    if not sample_submission.is_file():
        if not bool(competition.get("generate_template_if_missing", False)):
            raise FileNotFoundError(
                f"sample_submission.csv is missing: {sample_submission}. "
                "Upload the official template or set competition.generate_template_if_missing=true "
                "to create a schema-compatible template from test_ver2 row order."
            )
        sample_submission = run_directory / "generated_sample_submission.csv"
        pd.DataFrame(
            {"ncodpers": prepared_frame["ncodpers"].to_numpy(), "added_products": ""}
        ).to_csv(sample_submission, index=False)
    submission = run_competition_test_inference(
        prepared_frame, sample_submission, artifact_directories,
'''
if old not in source:
    raise RuntimeError('Expected submission call not found; no partial runtime patch applied.')
source = source.replace(old, new)
source = source.replace('"sample_submission": str(root / competition["sample_submission"]),', '"sample_submission": str(sample_submission),')
pipeline_file.write_text(source, encoding='utf-8')

run_dir = Path('/content/drive/MyDrive/projects/santander/data/artifacts/runs/lightgbm-acquisition-v1-20260920T192409Z-46a125e2')
config = yaml.safe_load((run_dir / 'config_resolved.yaml').read_text(encoding='utf-8'))
config['competition']['generate_template_if_missing'] = True
submission_config = run_dir / 'submission_config_resolved.yaml'
submission_config.write_text(yaml.safe_dump(config, sort_keys=False), encoding='utf-8')
for name in list(sys.modules):
    if name == 'src' or name.startswith('src.'):
        del sys.modules[name]
print({'submission_config': str(submission_config), 'fallback_template': True})

{'submission_config': '/content/drive/MyDrive/projects/santander/data/artifacts/runs/lightgbm-acquisition-v1-20260920T192409Z-46a125e2/submission_config_resolved.yaml', 'fallback_template': True}


In [ ]:
from pathlib import Path
root = Path('/content/drive/MyDrive/projects/santander/data')
matches = sorted(root.parent.rglob('*sample*submission*.csv'))
print('\n'.join(str(path) for path in matches) or 'NO_SAMPLE_SUBMISSION_CSV_FOUND')
print('raw CSV files:')
print('\n'.join(str(path.name) for path in sorted((root / 'raw').glob('*.csv'))))

NO_SAMPLE_SUBMISSION_CSV_FOUND
raw CSV files:
test_ver2.csv
train_ver2.csv


In [ ]:
# Replace slow row-wise Top-7 ranking, then rerun only the submission.
from pathlib import Path
import os
import sys
import traceback

repo = Path('/content/Santander-Product-Recommendation')
data_root = Path('/content/drive/MyDrive/projects/santander/data')
os.environ['SANTANDER_DATA_ROOT'] = str(data_root)
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

competition_file = repo / 'src/submission/competition.py'
source = competition_file.read_text(encoding='utf-8')
old = '''    ranked = score_frame.apply(lambda row: " ".join(row[row.ne(float("-inf"))].nlargest(top_k).index), axis=1)
    recommendations = pd.DataFrame({customer_id_column: prepared_input[customer_id_column].to_numpy(), "added_products": ranked.to_numpy()})
'''
new = '''    values = score_frame.to_numpy(copy=False)
    product_array = np.asarray(score_frame.columns, dtype=object)
    ranking = np.argsort(-values, axis=1, kind="stable")[:, :top_k]
    selected_scores = np.take_along_axis(values, ranking, axis=1)
    selected_products = product_array[ranking]
    ranked = [
        " ".join(products[np.isfinite(product_scores)])
        for products, product_scores in zip(selected_products, selected_scores, strict=True)
    ]
    recommendations = pd.DataFrame(
        {customer_id_column: prepared_input[customer_id_column].to_numpy(), "added_products": ranked}
    )
'''
if old not in source:
    raise RuntimeError('Expected slow ranking implementation was not found.')
if 'import numpy as np' not in source:
    source = source.replace('import pandas as pd\n', 'import pandas as pd\nimport numpy as np\n')
competition_file.write_text(source.replace(old, new), encoding='utf-8')

for name in list(sys.modules):
    if name == 'src' or name.startswith('src.'):
        del sys.modules[name]
from src.pipeline.lightgbm_v1 import run_lightgbm_v1_competition_from_config

run_dir = data_root / 'artifacts/runs/lightgbm-acquisition-v1-20260920T192409Z-46a125e2'
error_path = run_dir / 'submission_error.txt'
error_path.unlink(missing_ok=True)
try:
    submission = run_lightgbm_v1_competition_from_config(run_dir / 'model_artifacts.json')
    print({'submission': str(submission), 'exists': submission.is_file(), 'bytes': submission.stat().st_size if submission.is_file() else None})
except Exception:
    error_path.write_text(traceback.format_exc(), encoding='utf-8')
    raise

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[Submission 1/24] scoring ind_ahor_fin_ult1
[Submission 2/24] scoring ind_aval_fin_ult1
[Submission 3/24] scoring ind_cco_fin_ult1
[Submission 4/24] scoring ind_cder_fin_ult1
[Submission 5/24] scoring ind_cno_fin_ult1
[Submission 6/24] scoring ind_ctju_fin_ult1
[Submission 7/24] scoring ind_ctma_fin_ult1
[Submission 8/24] scoring ind_ctop_fin_ult1
[Submission 9/24] scoring ind_ctpp_fin_ult1
[Submission 10/24] scoring ind_deco_fin_ult1
[Submission 11/24] scoring ind_dela_fin_ult1
[Submission 12/24] scoring ind_deme_fin_ult1
[Submission 13/24] scoring ind_ecue_fin_ult1
[Submission 14/24] scoring ind_fond_fin_ult1
[Submission 15/24] scoring ind_hip_fin_ult1
[Submission 16/24] scoring ind_nom_pens_ult1
[Submission 17/24] scoring ind_nomina_ult1
[Submission 18/24] scoring ind_plan_fin_ult1
[Submission 19/24] scoring ind_pres_fin_ult1
[Submission 20/24] scoring ind_reca_fin_ult1
[Submission 21/24] scoring ind_recibo_ult1
[Submission 22/24] scoring ind_tjcr_fin_ult1
[Submission 23/24] scoring

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/projects/santander/data/raw/sample_submission.csv'

In [ ]:
# Diagnose and rerun competition submission only (no training).
from pathlib import Path
import os
import sys
import traceback

repo = Path('/content/Santander-Product-Recommendation')
data_root = Path('/content/drive/MyDrive/projects/santander/data')
os.environ['SANTANDER_DATA_ROOT'] = str(data_root)
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

# Emit a durable progress line before scoring each product model.
competition_file = repo / 'src/submission/competition.py'
source = competition_file.read_text(encoding='utf-8')
old = '    for product, directory in artifact_directories.items():\n        artifact, model = load_artifact(directory)\n'
new = '    for model_number, (product, directory) in enumerate(artifact_directories.items(), start=1):\n        print(f"[Submission {model_number}/24] scoring {product}", flush=True)\n        artifact, model = load_artifact(directory)\n'
if old in source:
    competition_file.write_text(source.replace(old, new), encoding='utf-8')
elif new not in source:
    raise RuntimeError('Could not install submission progress logging safely.')

for name in list(sys.modules):
    if name == 'src' or name.startswith('src.'):
        del sys.modules[name]

from src.pipeline.lightgbm_v1 import run_lightgbm_v1_competition_from_config
run_dir = data_root / 'artifacts/runs/lightgbm-acquisition-v1-20260920T192409Z-46a125e2'
error_path = run_dir / 'submission_error.txt'
error_path.unlink(missing_ok=True)
try:
    submission = run_lightgbm_v1_competition_from_config(run_dir / 'model_artifacts.json')
    print({'submission': str(submission), 'exists': submission.is_file(), 'bytes': submission.stat().st_size if submission.is_file() else None})
except Exception:
    error_path.write_text(traceback.format_exc(), encoding='utf-8')
    raise

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[Submission 1/24] scoring ind_ahor_fin_ult1
[Submission 2/24] scoring ind_aval_fin_ult1
[Submission 3/24] scoring ind_cco_fin_ult1
[Submission 4/24] scoring ind_cder_fin_ult1
[Submission 5/24] scoring ind_cno_fin_ult1
[Submission 6/24] scoring ind_ctju_fin_ult1
[Submission 7/24] scoring ind_ctma_fin_ult1
[Submission 8/24] scoring ind_ctop_fin_ult1
[Submission 9/24] scoring ind_ctpp_fin_ult1
[Submission 10/24] scoring ind_deco_fin_ult1
[Submission 11/24] scoring ind_dela_fin_ult1
[Submission 12/24] scoring ind_deme_fin_ult1
[Submission 13/24] scoring ind_ecue_fin_ult1
[Submission 14/24] scoring ind_fond_fin_ult1
[Submission 15/24] scoring ind_hip_fin_ult1
[Submission 16/24] scoring ind_nom_pens_ult1
[Submission 17/24] scoring ind_nomina_ult1
[Submission 18/24] scoring ind_plan_fin_ult1
[Submission 19/24] scoring ind_pres_fin_ult1
[Submission 20/24] scoring ind_reca_fin_ult1
[Submission 21/24] scoring ind_recibo_ult1
[Submission 22/24] scoring ind_tjcr_fin_ult1
[Submission 23/24] scoring

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/projects/santander/data/raw/sample_submission.csv'

# LightGBM acquisition baseline v1

Notebook này chỉ điều phối pipeline. Mỗi lần chạy bắt buộc đi qua `run_lightgbm_v1_from_config`, từ đó lưu config đã resolve, pipeline/model version, description, Git state, input/output fingerprints vào `lineage_manifest.json`.

## 1. Bootstrap

Chạy bootstrap chuẩn của project trước cell dưới đây.

In [ ]:
# ============================================================
# BOOTSTRAP: local + Colab CPU
#
# Local: d?ng tr?c ti?p source v? .env trong working tree.
# Colab browser: l?y config t? Colab Secrets.
# VS Code ? Colab CPU: d?n m?t COLAB_RUNTIME_CONFIG_B64 bundle duy nh?t.
# ============================================================
from __future__ import annotations

import base64
import binascii
import json
import os
import subprocess
import sys
from getpass import getpass
from pathlib import Path

from dotenv import load_dotenv

REPO_OWNER = "lbngyn"
REPO_NAME = "Santander-Product-Recommendation"
REPO_BRANCH = "feat/setup-pipeline"
REPO_URL = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"
RUNTIME_CONFIG_KEYS = (
    "GITHUB_TOKEN",
    "GCP_SERVICE_ACCOUNT_JSON",
    "GOOGLE_CLOUD_PROJECT",
    "GCS_BUCKET",
    "GCS_RAW_PREFIX",
    "GCS_CHECKPOINT_PREFIX",
)


def is_colab_runtime() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "src").is_dir() and (candidate / "requirements.txt").is_file():
            return candidate
    raise FileNotFoundError("Cannot find project root containing src/ and requirements.txt.")


def run_git(arguments: list[str], token: str) -> None:
    credentials = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    command = ["git", "-c", f"http.https://github.com/.extraheader=AUTHORIZATION: basic {credentials}", *arguments]
    subprocess.run(command, check=True)


def decode_runtime_bundle(encoded_value: str) -> dict[str, str]:
    try:
        payload = base64.b64decode(encoded_value, validate=True).decode("utf-8")
        config = json.loads(payload)
    except (binascii.Error, UnicodeDecodeError, json.JSONDecodeError) as error:
        raise ValueError("COLAB_RUNTIME_CONFIG_B64 must be a Base64-encoded JSON object.") from error
    missing = [key for key in RUNTIME_CONFIG_KEYS if not config.get(key)]
    if missing:
        raise ValueError(f"Runtime config bundle is missing: {', '.join(missing)}")
    if isinstance(config["GCP_SERVICE_ACCOUNT_JSON"], dict):
        config["GCP_SERVICE_ACCOUNT_JSON"] = json.dumps(config["GCP_SERVICE_ACCOUNT_JSON"])
    return {key: str(config[key]) for key in RUNTIME_CONFIG_KEYS}


def load_colab_ui_secrets() -> dict[str, str] | None:
    """Return secrets only when this kernel runs through the Colab browser UI."""
    try:
        from google.colab import userdata
        github_token = userdata.get("GITHUB_TOKEN")
        if not github_token:
            return None
        config = {"GITHUB_TOKEN": github_token}
        for key in RUNTIME_CONFIG_KEYS[1:]:
            value = userdata.get(key)
            if not value:
                raise RuntimeError(f"Missing Colab Secret: {key}")
            config[key] = value
        return config
    except Exception:
        return None


def load_colab_runtime_config() -> dict[str, str]:
    """Use browser Secrets, env bundle, or one secure runtime prompt."""
    ui_config = load_colab_ui_secrets()
    if ui_config:
        print("Using Colab Secrets.")
        return ui_config

    encoded_value = os.getenv("COLAB_RUNTIME_CONFIG_B64")
    if not encoded_value:
        encoded_value = getpass("Paste COLAB_RUNTIME_CONFIG_B64 once (runtime-only): ")
    return decode_runtime_bundle(encoded_value)


IS_COLAB = is_colab_runtime()
ENV = "colab" if IS_COLAB else "local"
os.environ["SANTANDER_RUNTIME"] = ENV

if IS_COLAB:
    runtime_config = load_colab_runtime_config()
    os.environ.update(runtime_config)
    github_token = runtime_config["GITHUB_TOKEN"]
    PROJECT_ROOT = Path("/content") / REPO_NAME

    if not PROJECT_ROOT.exists():
        print("Cloning source to Colab fast disk...")
        run_git(["clone", "--branch", REPO_BRANCH, REPO_URL, str(PROJECT_ROOT)], github_token)
    else:
        print("Syncing latest source to Colab fast disk...")

        subprocess.run([
            "git", "-C", str(PROJECT_ROOT), "config",
            "remote.origin.fetch", "+refs/heads/*:refs/remotes/origin/*"
        ], check=True)

        run_git(["-C", str(PROJECT_ROOT), "fetch", "origin", "--prune"], github_token)
        run_git([
            "-C", str(PROJECT_ROOT),
            "checkout", "-B", REPO_BRANCH,
            f"origin/{REPO_BRANCH}"
        ], github_token)

    requirements_path = PROJECT_ROOT / "requirements.txt"
    if not requirements_path.is_file():
        raise FileNotFoundError(f"requirements.txt is missing from branch {REPO_BRANCH}.")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements_path)], check=True)

    # Persist raw CSV and Parquet checkpoints between Colab runtimes.
    from google.colab import drive
    drive_mount = Path("/content/drive")
    if not (drive_mount / "MyDrive").exists():
        drive.mount(str(drive_mount))
    DATA_ROOT = drive_mount / "MyDrive/projects/santander/data"
else:
    PROJECT_ROOT = find_project_root(Path.cwd())
    load_dotenv(PROJECT_ROOT / ".env")
    DATA_ROOT = Path(os.getenv("SANTANDER_DATA_ROOT", PROJECT_ROOT / "data"))

# DuckDB temporary spill files should stay on Colab's fast ephemeral disk.
if IS_COLAB:
    os.environ["SANTANDER_DUCKDB_TEMP_DIRECTORY"] = "/content/santander_duckdb_temp"

os.environ["SANTANDER_DATA_ROOT"] = str(DATA_ROOT)
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Notebook ready | environment={ENV} | source={PROJECT_ROOT}")


Paste COLAB_RUNTIME_CONFIG_B64 once (runtime-only): ··········
Cloning source to Colab fast disk...
Mounted at /content/drive
Notebook ready | environment=colab | source=/content/Santander-Product-Recommendation


In [ ]:
from pathlib import Path
import sys

repo = Path('/content/Santander-Product-Recommendation')
prepare_file = repo / 'src/submission/prepare.py'
pipeline_file = repo / 'src/pipeline/lightgbm_v1.py'

text = prepare_file.read_text(encoding='utf-8')
text = text.replace(
    'def load_competition_input(path: str | Path, *, columns: Sequence[str]) -> object:\n',
    'def load_competition_input(path: str | Path, *, columns: Sequence[str], dtypes: dict[str, str] | None = None) -> object:\n',
)
old = '        return con.execute(f"SELECT {\', \'.join(_quote(column) for column in columns)} FROM read_parquet(?)", [str(path)]).fetchdf()\n'
new = '        frame = con.execute(f"SELECT {\', \'.join(_quote(column) for column in columns)} FROM read_parquet(?)", [str(path)]).fetchdf()\n        for column, dtype in (dtypes or {}).items():\n            frame[column] = frame[column].astype(dtype)\n        return frame\n'
if old not in text:
    raise RuntimeError('Expected competition input loader was not found.')
prepare_file.write_text(text.replace(old, new), encoding='utf-8')

text = pipeline_file.read_text(encoding='utf-8')
needle = '    feature_names = sorted({name for artifact in metadata.values() for name in artifact.schema.feature_names})\n'
insert = '''    expected_dtypes: dict[str, str] = {}
    for artifact in metadata.values():
        for name, dtype in artifact.schema.dtypes.items():
            if name in expected_dtypes and expected_dtypes[name] != dtype:
                raise ValueError(f"Conflicting artifact dtypes for {name!r}: {expected_dtypes[name]!r}, {dtype!r}")
            expected_dtypes[name] = dtype
'''
if needle not in text:
    raise RuntimeError('Expected feature-schema section was not found.')
text = text.replace(needle, needle + insert)
old_call = '    prepared_frame = load_competition_input(prepared, columns=list(dict.fromkeys(required_columns)))\n'
new_call = '    prepared_frame = load_competition_input(\n        prepared, columns=list(dict.fromkeys(required_columns)), dtypes=expected_dtypes,\n    )\n'
if old_call not in text:
    raise RuntimeError('Expected prepared-input call was not found.')
pipeline_file.write_text(text.replace(old_call, new_call), encoding='utf-8')

for name in list(sys.modules):
    if name == 'src' or name.startswith('src.'):
        del sys.modules[name]
print('Patched artifact dtype restoration for competition inference.')

Patched artifact dtype restoration for competition inference.


In [ ]:
# COLAB SOURCE UPDATE
# 1) Tr?n local: git add/commit/push source m?i l?n REPO_BRANCH.
# 2) Tr?n Colab: ch?y cell n?y, sau ?? ch?y l?i c?c cell import/EDA c?n d?ng code m?i.

if not IS_COLAB:
    print("Local environment: source is already current; no Colab sync is needed.")
else:
    status = subprocess.run(
        ["git", "-C", str(PROJECT_ROOT), "status", "--porcelain"],
        capture_output=True,
        text=True,
        check=True,
    ).stdout.strip()
    if status:
        raise RuntimeError(
            "The Colab source copy has uncommitted changes. Do not overwrite it; "
            "restart the runtime or resolve those changes first."
        )

    run_git(["-C", str(PROJECT_ROOT), "fetch", "origin", REPO_BRANCH], github_token)
    run_git(["-C", str(PROJECT_ROOT), "checkout", REPO_BRANCH], github_token)
    run_git(["-C", str(PROJECT_ROOT), "pull", "--ff-only", "origin", REPO_BRANCH], github_token)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_ROOT / "requirements.txt")],
        check=True,
    )

    import importlib
    importlib.invalidate_caches()
    stale_modules = [name for name in sys.modules if name == "src" or name.startswith("src.")]
    for name in stale_modules:
        del sys.modules[name]
    print(f"Colab source updated from {REPO_BRANCH}; cleared {len(stale_modules)} cached src modules. Re-run the import/config cell, then the desired EDA cells.")


Colab source updated from feat/setup-pipeline; cleared 0 cached src modules. Re-run the import/config cell, then the desired EDA cells.


In [ ]:
from pathlib import Path
import sys
import yaml

repo = Path('/content/Santander-Product-Recommendation')
model_file = repo / 'src/models/lightgbm_binary.py'
pipeline_file = repo / 'src/pipeline/lightgbm_v1.py'
config_file = repo / 'configs/baselines/lightgbm_v1.yaml'

text = model_file.read_text(encoding='utf-8')
text = text.replace(
    '    max_rows_per_product: int | None = None,\n',
    '    max_rows_per_product: int | None = None,\n    max_negative_rows_per_product: int | None = None,\n',
)
old = '''            query = f'SELECT {", ".join(_quote(c) for c in [*selected, "acq_" + product])} FROM read_parquet(?) WHERE previous_observation = 1 AND COALESCE({_quote("prev_" + product)}, 0) = 0'
            if max_rows_per_product:
                query += f" LIMIT {int(max_rows_per_product)}"
            frame = con.execute(query, [str(source)]).fetchdf()
'''
new = '''            projection = ", ".join(_quote(c) for c in [*selected, "acq_" + product])
            eligibility = f'previous_observation = 1 AND COALESCE({_quote("prev_" + product)}, 0) = 0'
            query = f'SELECT {projection} FROM read_parquet(?) WHERE {eligibility}'
            if max_rows_per_product:
                query += f" LIMIT {int(max_rows_per_product)}"
            elif max_negative_rows_per_product:
                label = _quote("acq_" + product)
                query = (
                    f'SELECT {projection} FROM read_parquet(?) WHERE {eligibility} AND {label} = 1 '
                    f'UNION ALL '
                    f'SELECT {projection} FROM (SELECT {projection} FROM read_parquet(?) '
                    f'WHERE {eligibility} AND {label} = 0 LIMIT {int(max_negative_rows_per_product)})'
                )
            parameters = [str(source), str(source)] if max_negative_rows_per_product and not max_rows_per_product else [str(source)]
            frame = con.execute(query, parameters).fetchdf()
'''
if old not in text:
    raise RuntimeError('Expected training query was not found; stop rather than applying a partial patch.')
model_file.write_text(text.replace(old, new), encoding='utf-8')

text = pipeline_file.read_text(encoding='utf-8')
needle = '            max_rows_per_product=model.get("max_rows_per_product"),\n'
if needle not in text:
    raise RuntimeError('Expected pipeline call was not found.')
pipeline_file.write_text(text.replace(needle, needle + '            max_negative_rows_per_product=model.get("max_negative_rows_per_product"),\n'), encoding='utf-8')

config = yaml.safe_load(config_file.read_text(encoding='utf-8'))
config['model']['max_rows_per_product'] = None
config['model']['max_negative_rows_per_product'] = 500_000
config['runtime']['memory_limit'] = '4GB'
config['runtime']['threads'] = 2
config_file.write_text(yaml.safe_dump(config, sort_keys=False), encoding='utf-8')

for name in list(sys.modules):
    if name == 'src' or name.startswith('src.'):
        del sys.modules[name]
print({'memory_limit': config['runtime']['memory_limit'], 'threads': config['runtime']['threads'], 'max_negative_rows_per_product': config['model']['max_negative_rows_per_product']})

{'memory_limit': '4GB', 'threads': 2, 'max_negative_rows_per_product': 500000}


## 2. Train and evaluate
Cập nhật `configs/baselines/lightgbm_v1.yaml` khi đổi experiment. Luôn tăng `pipeline.version` hoặc `model.version` và viết rõ `pipeline.description`; pipeline sẽ từ chối config thiếu hai trường bắt buộc. Đặt `SANTANDER_DATA_ROOT` tới thư mục chứa `interim/train.parquet`.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').is_dir():
    raise RuntimeError('Open this notebook with the repository root as the working directory.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
from src.pipeline.lightgbm_v1 import run_lightgbm_v1_from_config

load_dotenv(PROJECT_ROOT / '.env')
CONFIG_PATH = PROJECT_ROOT / 'configs' / 'baselines' / 'lightgbm_v1.yaml'
print(CONFIG_PATH.read_text(encoding='utf-8'))

pipeline:
  version: lightgbm-acquisition-v1
  description: 24 independent LightGBM binary acquisition classifiers. Customer profile
    values use month t; product ownership inputs use only t-1.
data:
  interim_train: interim/train.parquet
  model_panel: processed/lightgbm_v1/model_panel.parquet
  artifact_dir: artifacts
competition:
  raw_test: raw/test_ver2.csv
  interim_test: interim/test.parquet
  prepared_input: processed/lightgbm_v1/competition_test_input.parquet
  sample_submission: raw/sample_submission.csv
  history_date: 2016-05-28
  chunksize: 50000
  force_rebuild: false
  show_progress: true
  top_k: 7
features:
  force_process: false
model:
  version: lightgbm-binary-v1
  n_estimators: 20
  random_state: 42
  training_log_period: 5
  lightgbm_params:
    device_type: cpu
  max_rows_per_product: null
runtime:
  memory_limit: 10GB
  threads: 4
  temp_directory: data/.duckdb_tmp
tracking:
  mlflow:
    enabled: true
    tracking_uri: sqlite:///mlruns.db
    experiment_name:

In [ ]:
# Training all 24 models. Outputs are immutable per run_id.
# This cell is self-contained: after bootstrap it does not depend on
# the preceding import/config preview cell having been executed.
from pathlib import Path
import os
import sys

candidates = [Path.cwd().resolve(), Path('/content') / 'Santander-Product-Recommendation']
PROJECT_ROOT = next((path for path in candidates if (path / 'src').is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Project source is unavailable in this runtime. Run the bootstrap cell once after a runtime restart.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

if not os.environ.get('SANTANDER_DATA_ROOT'):
    colab_data_root = Path('/content/drive/MyDrive/projects/santander/data')
    os.environ['SANTANDER_DATA_ROOT'] = str(colab_data_root if colab_data_root.exists() else PROJECT_ROOT / 'data')

from src.pipeline.lightgbm_v1 import run_lightgbm_v1_from_config

CONFIG_PATH = PROJECT_ROOT / 'configs' / 'baselines' / 'lightgbm_v1.yaml'
result = run_lightgbm_v1_from_config(CONFIG_PATH)
result

[LightGBM 1/24] ind_ahor_fin_ult1 | eligible_rows=500,002, acquisitions=2 (0.0004%), features=32
[LightGBM] [Info] Number of positive: 2, number of negative: 500000
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.087476 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 724
[LightGBM] [Info] Number of data points in the train set: 500002, number of used features: 31
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.000004 -> initscore=-12.429216
[LightGBM] [Info] Start training from score -12.429216
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with po

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[LightGBM 3/24] ind_cco_fin_ult1 | eligible_rows=569,997, acquisitions=69,997 (12.2802%), features=32
[LightGBM] [Info] Number of positive: 69997, number of negative: 500000
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.084639 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 737
[LightGBM] [Info] Number of data points in the train set: 569997, number of used features: 31
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.122802 -> initscore=-1.966156
[LightGBM] [Info] Start training from score -1.966156
[5]	training's binary_logloss: 0.248102	training's auc: 0.93877
[10]	training's binary_logloss: 0.209247	training's auc: 0.94261
[15]	training's binary_logloss: 0.191173	training's auc: 0.944067
[20]	training's binary_logloss: 0.181717	training's auc: 0.945061
[LightGBM 3/24 complete] ind_cco_fin_ult1 | binary_logloss=0.181717, auc=0.945

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[LightGBM 14/24] ind_fond_fin_ult1 | eligible_rows=503,699, acquisitions=3,699 (0.7344%), features=32
[LightGBM] [Info] Number of positive: 3699, number of negative: 500000
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.082060 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 723
[LightGBM] [Info] Number of data points in the train set: 503699, number of used features: 30
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.007344 -> initscore=-4.906546
[LightGBM] [Info] Start training from score -4.906546
[5]	training's binary_logloss: 0.0355107	training's auc: 0.906479
[10]	training's binary_logloss: 0.0334891	training's auc: 0.911247
[15]	training's binary_logloss: 0.0323064	training's auc: 0.915367
[20]	training's binary_logloss: 0.0314835	training's auc: 0.917963
[LightGBM 14/24 complete] ind_fond_fin_ult1 | binary_logloss=0.031483, au

...

[LightGBM 23/24] ind_nom_pens_ult1 | eligible_rows=584,768, acquisitions=84,768 (14.4960%), features=32
[LightGBM] [Info] Number of positive: 84768, number of negative: 500000
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.083410 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 723
[LightGBM] [Info] Number of data points in the train set: 584768, number of used features: 30
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.144960 -> initscore=-1.774690
[LightGBM] [Info] Start training from score -1.774690
[5]	training's binary_logloss: 0.264975	training's auc: 0.95715
[10]	training's binary_logloss: 0.217056	training's auc: 0.957787
[15]	training's binary_logloss: 0.194156	training's auc: 0.958657
[20]	training's binary_logloss: 0.182148	training's auc: 0.959338
[LightGBM 23/24 complete] ind_nom_pens_ult1 | binary_logloss=0.182148, auc=

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[LightGBM 24/24] ind_recibo_ult1 | eligible_rows=653,205, acquisitions=153,205 (23.4544%), features=32
[LightGBM] [Info] Number of positive: 153205, number of negative: 500000
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.369280 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 725
[LightGBM] [Info] Number of data points in the train set: 653205, number of used features: 30
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.234544 -> initscore=-1.182831
[LightGBM] [Info] Start training from score -1.182831
[5]	training's binary_logloss: 0.429068	training's auc: 0.885913
[10]	training's binary_logloss: 0.382529	training's auc: 0.888861
[15]	training's binary_logloss: 0.360063	training's auc: 0.890243
[20]	training's binary_logloss: 0.348394	training's auc: 0.891431
[LightGBM 24/24 complete] ind_recibo_ult1 | binary_logloss=0.348394, auc=0.891431 | model_time=5.2s, estimated_remaining=0.0m


2026/09/20 19:26:02 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/09/20 19:26:02 INFO mlflow.store.db.utils: Updating database tables
2026/09/20 19:26:04 INFO mlflow.tracking.fluent: Experiment with name 'santander-lightgbm-acquisition' does not exist. Creating a new experiment.


{'run_id': 'lightgbm-acquisition-v1-20260920T192409Z-46a125e2',
 'description': '24 independent LightGBM binary acquisition classifiers. Customer profile values use month t; product ownership inputs use only t-1.',
 'pipeline_version': 'lightgbm-acquisition-v1',
 'model_version': 'lightgbm-binary-v1',
 'model_panel': '/content/drive/MyDrive/projects/santander/data/processed/lightgbm_v1/model_panel.parquet',
 'models': {'ind_ahor_fin_ult1': '/content/drive/MyDrive/projects/santander/data/artifacts/runs/lightgbm-acquisition-v1-20260920T192409Z-46a125e2/models/ind_ahor_fin_ult1',
  'ind_aval_fin_ult1': '/content/drive/MyDrive/projects/santander/data/artifacts/runs/lightgbm-acquisition-v1-20260920T192409Z-46a125e2/models/ind_aval_fin_ult1',
  'ind_cco_fin_ult1': '/content/drive/MyDrive/projects/santander/data/artifacts/runs/lightgbm-acquisition-v1-20260920T192409Z-46a125e2/models/ind_cco_fin_ult1',
  'ind_cder_fin_ult1': '/content/drive/MyDrive/projects/santander/data/artifacts/runs/lightg

In [ ]:
import json
from pathlib import Path

manifest = json.loads(Path(result['manifest']).read_text(encoding='utf-8'))
assert manifest['description']
assert manifest['pipeline_version'] == result['pipeline_version']
assert manifest['model_version'] == result['model_version']
manifest

In [ ]:
# Competition submission for the exact trained run.
# Reuses generic schema-validated predict() through the orchestration function.
from pathlib import Path
from src.pipeline.lightgbm_v1 import run_lightgbm_v1_competition_from_config

run_directory = Path(result['artifacts'])
submission = run_lightgbm_v1_competition_from_config(run_directory / 'model_artifacts.json')
print({'submission': str(submission), 'bytes': submission.stat().st_size})
submission